# 实战练习：使用标准 HuggingFace 栈进行 GRPO 训练

本节与 `section-06.ipynb` 实现相同的训练目标，但**不依赖 Unsloth 或 vLLM**，
完全使用标准 HuggingFace 生态（`transformers` + `peft` + `trl`）。

| 对比项 | section-06（Unsloth） | 本笔记（标准栈） |
|--------|----------------------|------------------|
| 模型加载 | `FastLanguageModel` | `AutoModelForCausalLM`（bfloat16 全精度） |
| LoRA | `FastLanguageModel.get_peft_model` | `peft.get_peft_model` |
| 梯度检查点 | unsloth 优化版 | 不启用（32GB 无需） |
| 推理 | `model.fast_generate`（vLLM） | `model.generate` |
| 模型保存 | `save_pretrained_merged` | `merge_and_unload` + `save_pretrained` |

> **适用场景**：标准环境（无法或不想安装 Unsloth/vLLM）、调试、理解原理。
> 代价是训练速度比 Unsloth 版慢约 2～3 倍。

## 环境安装

In [ ]:
# 安装标准 HuggingFace 栈
# transformers: 模型与 tokenizer
# peft:         LoRA 等参数高效微调
# trl:          GRPO / PPO 等强化学习微调
# accelerate:   分布式训练与混合精度
# datasets:     数据集加载
# wandb:        实验追踪与可视化
!pip install transformers peft trl accelerate datasets wandb
!pip install --upgrade pillow

## 加载模型（标准 HuggingFace 方式）

与 Unsloth 版不同，这里使用原生 `transformers` + `peft`：

1. `AutoModelForCausalLM.from_pretrained` — 以 bfloat16 全精度加载模型（32GB 无需量化）
2. `LoraConfig` + `get_peft_model` — 注入 LoRA 适配器

In [1]:
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# ---- 固定随机种子，保证可复现 ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- 训练配置 ----
MODEL_NAME  = "Qwen/Qwen2.5-7B-Instruct"  # 基础模型（7B，数学推理能力强）
LORA_RANK   = 32                           # LoRA 秩

# ---- 加载 Tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# decoder-only 模型生成时必须左填充，否则 attention mask 会截断生成
tokenizer.padding_side = "left"

# ---- 加载模型（全精度 bfloat16，RTX 5090 32GB）----
# Qwen2.5-7B 以 bfloat16 加载约占 14GB 显存，32GB 可容纳（剩余 ~18GB 用于激活值和优化器）
#
# attn_implementation="sdpa"：使用 PyTorch SDPA（Scaled Dot-Product Attention）。
#   RTX 5090 基于 Blackwell 架构（sm_120），flash-attn 2 对 sm_120 的支持尚不稳定，
#   建议使用 sdpa 作为安全选项。待 flash-attn 正式支持 sm_120 后可切换回
#   attn_implementation="flash_attention_2" 以获得额外 10~20% 速度提升。
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

# ---- LoRA 配置 ----
# lora_alpha = 2 * r（scaling=2）：比 scaling=1 收敛更快、训练更稳定
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,            # scaling = lora_alpha / r = 2
    target_modules=[
        # 注意力层
        "q_proj", "k_proj", "v_proj", "o_proj",
        # FFN 层
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.0,                    # GRPO 训练通常不加 dropout
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# ---- 应用 LoRA ----
model = get_peft_model(model, lora_config)

# ---- 不启用梯度检查点 ----
# 7B 模型占 ~14GB，剩余 ~18GB 足够存放激活值，无需以计算换显存。
# 开启梯度检查点反而会重新计算 forward activations，降低训练速度。

print("模型加载完成！")
model.print_trainable_parameters()  # 打印可训练参数量
print(f"模型设备：{next(model.parameters()).device}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

模型加载完成！
trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491
模型设备：cuda:0


## 数据准备

使用 **GSM8K** 数据集（小学数学题），训练目标是让模型以结构化推理格式作答。

### 目标格式

训练后模型应使用如下 XML 格式输出：

```
<thinking>
（逐步推理过程）
</thinking>

<answer>
（最终数字答案）
</answer>
```

这种格式受 DeepSeek-R1 启发，将推理链与最终答案分离，便于奖励函数验证。

In [2]:
# 定义系统提示词：指导模型使用 <thinking>/<answer> 格式回答
SYSTEM_PROMPT = """
Respond in the following format:

<thinking>
...
</thinking>

<answer>
...
</answer>

"""

print("系统提示词：")
print(SYSTEM_PROMPT)

系统提示词：

Respond in the following format:

<thinking>
...
</thinking>

<answer>
...
</answer>




In [3]:
import re
from datasets import load_dataset, Dataset

def extract_xml_answer(text: str) -> str:
    """
    从 <answer>...</answer> 标签中提取答案
    用于从模型输出中解析最终答案
    """
    answer = text.split("<answer>")[-1]    # 取最后一个 <answer> 之后的内容
    answer = answer.split("</answer>")[0]  # 取 </answer> 之前的内容
    return answer.strip()


def extract_hash_answer(text: str) -> str | None:
    """
    从 GSM8K 格式的答案中提取数字答案
    GSM8K 答案格式：「...计算步骤... #### 答案数字」
    """
    if "####" not in text:
        return None  # 格式不匹配，返回 None
    return text.split("####")[1].strip()  # 提取 #### 后的数字


def get_gsm8k_questions(split="train") -> Dataset:
    """
    加载并预处理 GSM8K 数学数据集

    将原始数据转换为包含以下字段的格式：
    - prompt: 带系统提示的消息列表（供模型生成回答）
    - answer: 正确的数字答案（供奖励函数验证）
    """
    # 加载 GSM8K 数据集（main 配置包含 CoT 格式的答案）
    data = load_dataset("openai/gsm8k", "main")[split]

    # 转换数据格式
    data = data.map(
        lambda x: {
            # 构造消息格式（包含系统提示和用户问题）
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},  # 系统提示
                {"role": "user",   "content": x["question"]},  # 用户问题
            ],
            # 提取标准答案（只保留数字部分）
            "answer": extract_hash_answer(x["answer"]),
        }
    )
    return data


# 加载数据集
dataset = get_gsm8k_questions()

print(f"数据集大小：{len(dataset)}")
print()
print("第一个样本：")
print(f"问题：{dataset[0]['prompt'][1]['content']}")
print(f"正确答案：{dataset[0]['answer']}")

数据集大小：7473

第一个样本：
问题：Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
正确答案：72


## 定义奖励函数

本练习使用**多个奖励函数的组合**，从不同角度引导模型：

| 奖励函数 | 目的 | 最高分 |
|----------|------|--------|
| `correctness_reward_func` | 答案正确性 | +2.0 |
| `int_reward_func` | 鼓励数字答案 | +0.5 |
| `strict_format_reward_func` | 严格 XML 格式 | +0.5 |
| `soft_format_reward_func` | 宽松 XML 格式 | +0.5 |
| `xmlcount_reward_func` | XML 标签完整性 | +0.5 |
| **合计** | | **+4.0** |

**设计哲学**：正确性权重最高（2.0），格式奖励在训练早期帮助模型学习结构，
后期主要靠正确性奖励驱动。

In [4]:
import re

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    正确性奖励函数：最重要的奖励，验证答案是否正确

    答案正确：+2.0（权重最高）
    答案错误：+0.0
    """
    # 提取每个 completion 的文本内容
    responses = [completion[0]["content"] for completion in completions]
    q = prompts[0][-1]["content"]  # 获取问题文本（用于调试输出）

    # 从每个回答中提取 <answer>...</answer> 内的内容
    extracted_responses = [extract_xml_answer(r) for r in responses]

    # 打印调试信息（训练时建议注释掉：batch_size=4×num_generations=8，
    # 每步会触发 32 次 print，会明显拖慢训练速度）
    # print(
    #     "-" * 20,
    #     f"问题：\n{q}",
    #     f"\n正确答案：\n{answer[0]}",
    #     f"\n模型回答：\n{responses[0]}",
    #     f"\n提取到的答案：\n{extracted_responses[0]}",
    # )

    # 对比每个提取答案与正确答案，给予奖励
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]


def int_reward_func(completions, **kwargs) -> list[float]:
    """
    整数奖励函数：鼓励模型给出数字（而非文字）答案

    答案是纯数字：+0.5
    答案包含非数字字符：+0.0
    """
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]

    # isdigit() 检查字符串是否全为数字字符
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]


def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """
    严格格式奖励函数：检查是否完全符合期望格式
    格式要求：\n<thinking>\n...\n</thinking>\n\n<answer>\n...\n</answer>\n

    完全符合格式：+0.5
    不符合：+0.0
    """
    # 严格的正则表达式（要求精确的换行符位置）
    pattern = r"^\n<thinking>\n.*?\n</thinking>\n\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]


def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """
    宽松格式奖励函数：只要包含基本的 XML 标签结构即可
    格式要求：包含 <thinking>...</thinking> 和 <answer>...</answer>

    包含基本标签结构：+0.5
    缺少标签：+0.0
    """
    # 宽松的正则表达式（允许标签间有任意空白）
    pattern = r"<thinking>.*?</thinking>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]


def count_xml(text) -> float:
    """
    XML 标签计数函数：计算各 XML 标签的完整性

    每个正确的标签 +0.125 分（共 4 个标签，满分 0.5）
    额外惩罚：</answer> 之后若有多余内容，每个字符 -0.001 分
    （防止模型在答案后添加无关内容）
    """
    count = 0.0

    # 检查开始标签：<thinking>\n 出现恰好一次
    if text.count("<thinking>\n") == 1:
        count += 0.125

    # 检查结束标签：\n</thinking>\n 出现恰好一次
    if text.count("\n</thinking>\n") == 1:
        count += 0.125

    # 检查答案开始标签：\n<answer>\n 出现恰好一次
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        # 惩罚 </answer> 之后的多余内容
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001

    # 检查答案结束标签：\n</answer> 出现恰好一次
    if text.count("\n</answer>") == 1:
        count += 0.125
        # 惩罚 </answer> 之后的多余内容（不计最后的换行符）
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001

    return count


def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    """
    XML 标签完整性奖励函数：用 count_xml 评估 XML 格式完整性
    满分 0.5，鼓励精确的格式使用
    """
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]


print("奖励函数定义完成！")
print()
print("各奖励函数说明：")
print(f"  correctness_reward_func:    正确答案 → +2.0（最重要）")
print(f"  int_reward_func:            纯数字答案 → +0.5")
print(f"  strict_format_reward_func:  严格格式 → +0.5")
print(f"  soft_format_reward_func:    宽松格式 → +0.5")
print(f"  xmlcount_reward_func:       XML 完整性 → 最高 +0.5")
print(f"  总计最高得分：+4.0")

奖励函数定义完成！

各奖励函数说明：
  correctness_reward_func:    正确答案 → +2.0（最重要）
  int_reward_func:            纯数字答案 → +0.5
  strict_format_reward_func:  严格格式 → +0.5
  soft_format_reward_func:    宽松格式 → +0.5
  xmlcount_reward_func:       XML 完整性 → 最高 +0.5
  总计最高得分：+4.0


## 配置并启动 GRPO 训练

### `generation_batch_size` 约束

TRL 要求：`generation_batch_size % num_generations == 0`

其中 `generation_batch_size` 默认 = `per_device_train_batch_size × gradient_accumulation_steps`。

本例（5090 32GB + Qwen2.5-7B 配置）：
- `per_device_train_batch_size = 4`
- `gradient_accumulation_steps = 4`
- `generation_batch_size = 4 × 4 = 16`
- `num_generations = 8`
- 约束验证：`16 % 8 == 0` ✓

### 显存分配估算

| 用途 | 显存占用 |
|------|---------|
| 模型权重（7B × bf16） | ~14 GB |
| LoRA 适配器 + 优化器状态 | ~1.5 GB |
| 激活值（batch=4, seq=512） | ~4 GB |
| 生成阶段缓冲 | ~3 GB |
| **合计** | **~22.5 GB** |

32GB 显存留有约 9GB 余量，足够安全运行。

## 实验追踪（W&B）

使用 [Weights & Biases](https://wandb.ai) 记录训练曲线、奖励函数得分和超参数。

首次运行需要登录：
```bash
wandb login
# 或在 notebook 中：
# import wandb; wandb.login()
```

In [5]:
import wandb
from datetime import datetime

# W&B 项目与运行命名：按「模型名_数据集名」格式
WANDB_PROJECT  = f"{MODEL_NAME.split('/')[-1]}-gsm8k"   # qwen2.5-7b-instruct-gsm8k
WANDB_RUN_NAME = f"{MODEL_NAME.split('/')[-1]}-grpo-{datetime.now().strftime('%m%d-%H%M')}"

# 初始化 W&B run（在 GRPOTrainer 之前调用，确保 trainer 能正确接管）
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model":                       MODEL_NAME,
        "dataset":                     "openai/gsm8k",
        "lora_rank":                   LORA_RANK,
        "lora_alpha":                  LORA_RANK * 2,
        "num_generations":             8,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "max_completion_length":       512,
        "learning_rate":               5e-6,
        "max_steps":                   250,
        "loss_type":                   "grpo",
    },
)

print(f"W&B 项目：{WANDB_PROJECT}")
print(f"运行名称：{WANDB_RUN_NAME}")
print(f"W&B URL：{wandb.run.get_url()}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: goosmanlei (goosmanlei-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


W&B 项目：Qwen2.5-7B-Instruct-gsm8k
运行名称：Qwen2.5-7B-Instruct-grpo-0324-1834
W&B URL：https://wandb.ai/goosmanlei-personal/Qwen2.5-7B-Instruct-gsm8k/runs/6gooun9g


In [6]:
from trl import GRPOConfig, GRPOTrainer

num_generations = 8     # 每题生成候选数

# generation_batch_size 约束：必须能被 num_generations 整除
# generation_batch_size = per_device_train_batch_size * gradient_accumulation_steps
# 4 * 4 = 16，16 % 8 == 0 ✓

training_args = GRPOConfig(
    # ---- 优化器参数 ----
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",
    bf16=True,                           # 显式启用 bfloat16 混合精度（5090 完整支持 BF16）

    # ---- 批次参数（5090 32GB + 7B 模型配置）----
    # 7B 模型占 ~14GB，剩余 ~18GB；batch_size 从 8 降到 4 以留出激活值空间
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # 4*4=16，满足可被 num_generations(8) 整除的约束

    # ---- GRPO 核心参数 ----
    num_generations=num_generations,
    loss_type="grpo",
    # GSM8K 数学题推理链无需过长，512 足够；避免生成过长拖慢训练速度。
    max_completion_length=512,

    # ---- 训练控制 ----
    max_steps=250,
    save_steps=250,
    max_grad_norm=0.1,

    # ---- 日志 ----
    logging_steps=1,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
    output_dir="outputs",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,          # XML 完整性（0.5）
        soft_format_reward_func,        # 宽松格式（0.5）
        strict_format_reward_func,      # 严格格式（0.5）
        int_reward_func,               # 整数答案（0.5）
        correctness_reward_func,        # 正确性（2.0，最重要）
    ],
    args=training_args,
    train_dataset=dataset,
)

print("GRPOTrainer 初始化完成！")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


GRPOTrainer 初始化完成！


In [7]:
# 开始训练！
# RTX 5090 32GB + Qwen2.5-7B 全精度，250 步约需 45～90 分钟
# 注意：前 150～200 步可能看不到明显的奖励提升，这是正常的，请耐心等待！
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
1,0.000000
2,-0.000000
3,-0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,-0.000000
9,0.000000
10,-0.000000


TrainOutput(global_step=250, training_loss=-1.1359155178070069e-07, metrics={'train_runtime': 1700.1662, 'train_samples_per_second': 2.353, 'train_steps_per_second': 0.147, 'total_flos': 0.0, 'train_loss': -1.1359155178070069e-07})

> **提示**：训练初期（前 150～200 步）奖励接近 0 是正常的——模型需要时间学习格式和推理模式。
> 当 `correctness_reward_func` 开始出现非零值时，说明训练进入了有效阶段。

## 测试训练好的模型

In [8]:
# 保存 LoRA 权重（PEFT 格式，只保存 adapter 参数，文件很小）
model.save_pretrained("grpo_saved_lora")
tokenizer.save_pretrained("grpo_saved_lora")
print("LoRA 权重已保存到 grpo_saved_lora/")

LoRA 权重已保存到 grpo_saved_lora/


In [19]:
# 准备测试问题（注意：这是一个开放性问题，非 GSM8K 格式）
test_question = "Calculate the value of π to 20 decimal places."

# 切换到推理模式
model.eval()

# 获取实际设备（PEFT + device_map 下 model.device 不可靠，需从参数推断）
device = next(model.parameters()).device

# 应用 chat template 格式化输入
# apply_chat_template 返回 BatchEncoding 对象，需提取 .input_ids 获取纯 tensor
inputs = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": test_question},
    ],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(device)
input_ids = inputs.input_ids  # 提取 tensor，供 generate 和解码切片使用

# 使用标准 HuggingFace generate 推理
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,   # 避免 open-end 生成警告
        eos_token_id=tokenizer.eos_token_id,   # 遇到 eos 立即停止
    )

# 只解码新生成的 token（去掉 prompt 部分）
output = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

In [20]:
# 对比：基模（未训练）的输出
# 临时禁用 LoRA adapter，让模型退回到原始基模权重，无需重新加载
model.disable_adapter_layers()

with torch.no_grad():
    base_outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

base_output = tokenizer.decode(base_outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

# 恢复 LoRA adapter
model.enable_adapter_layers()

print(f"问题：{test_question}")
print()
print("=" * 40)
print("基模输出（未经 GRPO 训练）：")
print("=" * 40)
print(base_output)
print()
print("=" * 40)
print("训练后输出（GRPO 微调）：")
print("=" * 40)
print(output)
print()
print("观察：基模是否也使用了 <thinking>/<answer> 格式？GRPO 训练带来了哪些变化？")

问题：Calculate the value of π to 20 decimal places.

基模输出（未经 GRPO 训练）：
<thinking>
To calculate π to 20 decimal places, we can use various algorithms or look up a reliable source. One such source is Wolfram Alpha or similar mathematical databases that provide precise values of mathematical constants.
</thinking>

<answer>
3.14159265358979323846
</answer>

Note: The value provided is accurate to 20 decimal places as requested.

训练后输出（GRPO 微调）：
<thinking>
To calculate π to 20 decimal places, we can use various algorithms such as the Bailey–Borwein–Plouffe (BBP) formula or the Chudnovsky algorithm. However, for simplicity and precision, I will refer to a reliable source that has computed π to many decimal places.

The value of π to 20 decimal places is 3.14159265358979323846.
</thinking>

<answer>
3.14159265358979323846
</answer>

观察：基模是否也使用了 <thinking>/<answer> 格式？GRPO 训练带来了哪些变化？


## 保存与发布模型

标准 HuggingFace PEFT 提供两种保存方式：

| 方式 | 说明 | 适用场景 |
|------|------|----------|
| 只保存 LoRA adapter | 文件很小（几十 MB） | 继续微调、与原始基模分开管理 |
| 合并后保存完整模型 | 文件较大（原始大小） | 部署推理、不依赖原始基模 |

In [13]:
# --------------------------------------------------------
# 方式 1：只保存 LoRA adapter（文件极小，推荐用于继续训练）
# --------------------------------------------------------
model.save_pretrained("grpo_saved_lora")
tokenizer.save_pretrained("grpo_saved_lora")
print("LoRA adapter 已保存到 grpo_saved_lora/")
print()

# --------------------------------------------------------
# 方式 2：合并 LoRA 权重并保存完整模型（全精度，可直接部署）
# 注：merge_and_unload 只在非量化模型上可靠工作；
#     本笔记使用 bfloat16 全精度加载，可直接合并。
# --------------------------------------------------------
merged_model = model.merge_and_unload()
merged_model.save_pretrained("model_merged", safe_serialization=True)
tokenizer.save_pretrained("model_merged")
print("已合并并保存完整模型到 ./model_merged/")

# --------------------------------------------------------
# 方式 3：推送到 HuggingFace Hub
# --------------------------------------------------------
merged_model.push_to_hub(
    "goosmanlei/Qwen2.5-7B-Instruct-GRPO-GSM8K",
)
tokenizer.push_to_hub(
    "goosmanlei/Qwen2.5-7B-Instruct-GRPO-GSM8K",
)

LoRA adapter 已保存到 grpo_saved_lora/



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

已合并并保存完整模型到 ./model_merged/


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/goosmanlei/Qwen2.5-7B-Instruct-GRPO-GSM8K/commit/276d78edd9e3cff4ec99d176cfedc20b0b17d70c', commit_message='Upload tokenizer', commit_description='', oid='276d78edd9e3cff4ec99d176cfedc20b0b17d70c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/goosmanlei/Qwen2.5-7B-Instruct-GRPO-GSM8K', endpoint='https://huggingface.co', repo_type='model', repo_id='goosmanlei/Qwen2.5-7B-Instruct-GRPO-GSM8K'), pr_revision=None, pr_num=None)

## 本节小结

### 与 Unsloth 版（section-06）的关键差异

| 方面 | Unsloth 版 | 本版（标准栈） |
|------|------------|----------------|
| 安装依赖 | unsloth, vllm | transformers, peft, trl, accelerate |
| 模型加载 | `FastLanguageModel` | `AutoModelForCausalLM`（bfloat16 全精度） |
| 量化 | 内置简化 | 不量化（32GB 可直接加载 7B） |
| LoRA 注入 | `get_peft_model`（内部封装） | `LoraConfig` + `get_peft_model` |
| 梯度检查点 | 自定义优化版 | 不启用（显存充足，无需牺牲速度） |
| 生成速度 | vLLM 加速（快 2～3x） | 标准 `model.generate` |
| 模型保存 | `save_pretrained_merged` | `merge_and_unload` + `save_pretrained` |
| 兼容性问题 | vllm_ascend / mergekit bug | 无（标准库） |

### 你学到了什么

1. **全精度加载**：显存充足时直接用 bfloat16，避免量化引入的额外复杂度
2. **PEFT LoRA**：`LoraConfig` 的关键参数（`r`、`lora_alpha`、`target_modules`）
3. **generation_batch_size 约束**：`gradient_accumulation_steps × batch_size` 必须能被 `num_generations` 整除
4. **PEFT 保存方式**：adapter-only vs. `merge_and_unload` 的取舍

### 参考资料

- [HuggingFace PEFT 文档](https://huggingface.co/docs/peft)
- [TRL GRPO 文档](https://huggingface.co/docs/trl/main/en/grpo_trainer)
- [Qwen2.5-7B-Instruct 模型页面](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)